# Modern Sarvam-1 Fine-Tuning Notebook

This notebook uses the latest versions of HuggingFace libraries to ensure compatibility with Python 3.13.
Run all cells from top to bottom.

In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl liger-kernel

In [ ]:
import torch
import json
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "sarvamai/sarvam-1"
OUTPUT_DIR = "sarvam-finetuned"

data = [
    {"messages": [{"role": "system", "content": "You are a helpful real estate agent. Answer queries concisely."}, {"role": "user", "content": "I am looking for a 2BHK apartment in Bangalore."}, {"role": "assistant", "content": "Great! Are you looking to buy or rent? And what is your preferred budget and location in Bangalore?"}]},
    {"messages": [{"role": "system", "content": "You are a helpful real estate agent. Answer queries concisely."}, {"role": "user", "content": "Hi, I want to rent a 3BHK in Indiranagar, my budget is around 60k."}, {"role": "assistant", "content": "I can certainly help you with that. A 3BHK in Indiranagar for 60k is a bit tight but possible. Are you open to nearby areas like CV Raman Nagar as well?"}]},
    {"messages": [{"role": "system", "content": "You are a helpful real estate agent. Answer queries concisely."}, {"role": "user", "content": "What is the current rate per sqft in Whitefield?"}, {"role": "assistant", "content": "The current rate in Whitefield varies depending on the builder and amenities, but it generally ranges from ₹7,000 to ₹12,000 per sq ft. Are you looking for a ready-to-move-in property?"}]}
]

with open("real_estate_dataset.jsonl", "w") as f:
    for item in data:
        f.write(json.dumps(item) + "\n")

def format_instruction(example):
    prompt = ""
    for msg in example["messages"]:
        if msg["role"] == "system":
            prompt += f"System: {msg['content']}\n"
        elif msg["role"] == "user":
            prompt += f"User: {msg['content']}\n"
        elif msg["role"] == "assistant":
            prompt += f"Assistant: {msg['content']}\n"
    return {"text": prompt}

dataset = load_dataset("json", data_files="real_estate_dataset.jsonl", split="train")
dataset = dataset.map(format_instruction)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    max_steps=50, 
    save_steps=25,
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Fine-tuning complete! You can now zip and download the folder.")

In [ ]:
!zip -r sarvam-finetuned.zip sarvam-finetuned
from google.colab import files
files.download("sarvam-finetuned.zip")